In [ ]:
import os
import torch
import sys
sys.path.append("src")
os.chdir("..")

from dataset import load_splits, IUXrayDataset
from model import MedCLIP
from torch.utils.data import DataLoader

DEVICE = "mps"

_, _, test_df = load_splits()
test_df = test_df.reset_index(drop=True)
ds = IUXrayDataset(test_df)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)


/Users/sriya/Desktop/College/Deep_Learning/Final_Project/medclip-iu/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 3060  Val: 383  Test: 383


In [5]:
def get_embs(ckpt):
    m = MedCLIP().to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    imgs, txts = [], []
    with torch.no_grad():
        for b in loader:
            i, t = m.get_embeddings(
                b["pixel_values"].to(DEVICE),
                b["input_ids"].to(DEVICE),
                b["attention_mask"].to(DEVICE)
            )
            imgs.append(i.cpu())
            txts.append(t.cpu())
    return torch.cat(imgs), torch.cat(txts)

print("Loading checkpoints...")
base_i, base_t = get_embs("results/baseline_best.pt")
mask_i, mask_t = get_embs("results/masked_best.pt")

#find interesting queries, ones where the two models disagree
#specifically: masked gets a semantic hit at rank 1, baseline doesn't
def shares_label(a, b):
    sa = set(x.strip() for x in a.split("|") if x.strip())
    sb = set(x.strip() for x in b.split("|") if x.strip())
    return bool(sa & sb)

print("\nSearching for contrastive examples (masked wins, baseline doesn't)...\n")

good_queries = []
for q in range(len(test_df)):
    query_mesh = test_df.iloc[q]["mesh_labels"]

    # baseline top-1
    base_sim = base_i[q] @ base_t.T
    base_top1 = base_sim.topk(1).indices.item()
    base_hit = shares_label(query_mesh, test_df.iloc[base_top1]["mesh_labels"])

    # masked top-1
    mask_sim = mask_i[q] @ mask_t.T
    mask_top1 = mask_sim.topk(1).indices.item()
    mask_hit = shares_label(query_mesh, test_df.iloc[mask_top1]["mesh_labels"])

    if mask_hit and not base_hit:
        good_queries.append(q)

print(f"Found {len(good_queries)} queries where masked wins at R@1 but baseline doesn't\n")

#print top 5 of these
for q in good_queries[:5]:
    query_mesh = test_df.iloc[q]["mesh_labels"]
    query_text = test_df.iloc[q]["text"]

    print("=" * 70)
    print(f"QUERY idx={q}")
    print(f"  MeSH:   {query_mesh}")
    print(f"  Report: {query_text[:120]}")
    print()

    for name, img_e, txt_e in [("BASELINE", base_i, base_t), ("MASKED", mask_i, mask_t)]:
        sim = img_e[q] @ txt_e.T
        top3 = sim.topk(3).indices.tolist()
        print(f"  {name} top-3 retrievals:")
        for rank, idx in enumerate(top3):
            hit = "+" if shares_label(query_mesh, test_df.iloc[idx]["mesh_labels"]) else "-"
            exact = " [EXACT MATCH]" if idx == q else ""
            print(f"    {rank+1}. {hit} [{test_df.iloc[idx]['mesh_labels'][:100]}]{exact}")
            print(f"       {test_df.iloc[idx]['text'][:100]}")
    print()

Loading checkpoints...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 65106.59it/s]



Searching for contrastive examples (masked wins, baseline doesn't)...

Found 53 queries where masked wins at R@1 but baseline doesn't

QUERY idx=6
  MeSH:   Cardiomegaly/mild|Markings/lung/interstitial/diffuse/mild|Spine/degenerative|Heart Failure/mild
  Report: 1. Findings consistent with mild congestive heart failure.

  BASELINE top-3 retrievals:
    1. - [Surgical Instruments/lung/hilum/right|Volume Loss/lung/right|Density/lung/lower lobe/left]
       Increasing density in the superior segment XXXX the left lower lobe, XXXX seen on lateral view, cons
    2. + [Cardiomegaly/mild|Implanted Medical Device/humerus/right]
       Stable mild cardiomegaly without acute cardiopulmonary abnormality.
    3. - [Implanted Medical Device/left|Cardiac Shadow/enlarged|Cardiomegaly]
       Stable cardiomegaly without acute cardiopulmonary disease.
  MASKED top-3 retrievals:
    1. + [Cardiomegaly/mild|Implanted Medical Device/aortic valve|Spine/degenerative]
       No acute cardiopulmonary diseas